# Diabetes Risk Assessment
## Logistic Regression + Bayesian Network

This notebook contains the complete implementation, including:
- Table 1: Baseline characteristics
- Table 2: Training vs testing distribution
- Data preprocessing and Random Forest iterative imputation
- Logistic Regression and ROC-AUC evaluation
- Bayesian Network discretization, BIC scoring, structure learning, CPTs and inference
- ROC comparison, network visualization, model comparison, and personalized risk assessment

**Notebook file:** `Diabetes_Risk_Assessment_Full_with_Table1_Table2.ipynb`

In [2]:
from google.colab import files
uploaded = files.upload()

Saving diabetes.csv to diabetes.csv


In [4]:

# ============================================================
# DIABETES RISK ASSESSMENT
# TABLE 1: BASELINE CHARACTERISTICS
# TABLE 2: TRAINING VS TESTING DISTRIBUTION
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.model_selection import train_test_split
from IPython.display import display

# ============================================================
# 1. IMPORT DATASET
# ============================================================

df = pd.read_csv("/content/diabetes.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

# ============================================================
# 2. TABLE 1: BASELINE CHARACTERISTICS
# ============================================================

def generate_table_1(df):

    print("\n" + "=" * 80)
    print("TABLE 1: BASELINE CHARACTERISTICS")
    print("=" * 80)

    non_diabetes = df[df["Outcome"] == 0]
    diabetes = df[df["Outcome"] == 1]

    print("Non-diabetes patients:", len(non_diabetes))
    print("Diabetes patients:", len(diabetes))

    variables = {
        "Pregnancy": "Pregnancies",
        "Glucose": "Glucose",
        "BP": "BloodPressure",
        "Skin": "SkinThickness",
        "Insulin": "Insulin",
        "BMI": "BMI",
        "Pedigree": "DiabetesPedigreeFunction",
        "Age": "Age"
    }

    results = []

    for display_name, column in variables.items():

        group_0 = non_diabetes[column].dropna()
        group_1 = diabetes[column].dropna()

        # Median and quartiles
        median_0 = group_0.median()
        q1_0 = group_0.quantile(0.25)
        q3_0 = group_0.quantile(0.75)

        median_1 = group_1.median()
        q1_1 = group_1.quantile(0.25)
        q3_1 = group_1.quantile(0.75)

        # Mann-Whitney U test
        _, p_value = mannwhitneyu(
            group_0,
            group_1,
            alternative="two-sided"
        )

        p_text = (
            "<0.001" if p_value < 0.001
            else f"{p_value:.3f}"
        )

        results.append({
            "Variables": display_name,
            "Non-diabetes": (
                f"{median_0:.2f} "
                f"({q1_0:.2f}, {q3_0:.2f})"
            ),
            "Diabetes": (
                f"{median_1:.2f} "
                f"({q1_1:.2f}, {q3_1:.2f})"
            ),
            "p": p_text
        })

    table_1 = pd.DataFrame(results)

    display(table_1)

    table_1.to_csv(
        "Table_1_Baseline_Characteristics.csv",
        index=False
    )

    print("Table 1 saved successfully.")

    return table_1


table_1 = generate_table_1(df)


# ============================================================
# 3. SPLIT DATA FOR TABLE 2
# ============================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["Outcome"]
)

print("\nTraining set size:", len(train_df))
print("Testing set size:", len(test_df))


# ============================================================
# 4. TABLE 2: TRAINING VS TESTING DISTRIBUTION
# ============================================================

def generate_table_2(train_df, test_df):

    print("\n" + "=" * 80)
    print("TABLE 2: TRAINING VS TESTING DISTRIBUTION")
    print("=" * 80)

    def categorize_data(data):

        result = pd.DataFrame(index=data.index)

        result["Pregnancies"] = pd.cut(
            data["Pregnancies"],
            [-np.inf, 3, 6, np.inf],
            labels=["Low", "Medium", "High"]
        )

        result["Glucose"] = pd.cut(
            data["Glucose"],
            [-np.inf, 100, 140, np.inf],
            labels=["Low", "Medium", "High"]
        )

        result["Blood Pressure"] = pd.cut(
            data["BloodPressure"],
            [-np.inf, 60, 90, np.inf],
            labels=["Low", "Normal", "High"]
        )

        result["Skin Thickness"] = pd.cut(
            data["SkinThickness"],
            [-np.inf, 20, 30, np.inf],
            labels=["Low", "Medium", "High"]
        )

        result["Insulin"] = pd.cut(
            data["Insulin"],
            [-np.inf, 100, 200, np.inf],
            labels=["Low", "Medium", "High"]
        )

        result["BMI"] = pd.cut(
            data["BMI"],
            [-np.inf, 18.5, 25, np.inf],
            labels=["Underweight", "Normal", "Overweight"]
        )

        result["Pedigree"] = pd.cut(
            data["DiabetesPedigreeFunction"],
            [-np.inf, 0.30, 0.70, np.inf],
            labels=["Low", "Middle", "High"]
        )

        result["Age"] = pd.cut(
            data["Age"],
            [-np.inf, 35, 65, np.inf],
            labels=["Young", "Middle", "Senior"]
        )

        result["Diabetes"] = data["Outcome"].map({
            0: "No",
            1: "Yes"
        })

        return result

    train_cat = categorize_data(train_df)
    test_cat = categorize_data(test_df)

    variables = [
        "Pregnancies",
        "Glucose",
        "Blood Pressure",
        "Skin Thickness",
        "Insulin",
        "BMI",
        "Pedigree",
        "Age",
        "Diabetes"
    ]

    results = []

    for variable in variables:

        train_values = train_cat[variable]
        test_values = test_cat[variable]

        # Combine the categories from both sets
        levels = list(
            dict.fromkeys(
                list(train_values.dropna().unique())
                + list(test_values.dropna().unique())
            )
        )

        # Build the contingency table
        contingency = []

        for level in levels:
            train_count = (train_values == level).sum()
            test_count = (test_values == level).sum()

            contingency.append([train_count, test_count])

        contingency = np.array(contingency)

        # Chi-square test
        _, p_value, _, _ = chi2_contingency(contingency)

        p_text = (
            "<0.001" if p_value < 0.001
            else f"{p_value:.3f}"
        )

        for i, level in enumerate(levels):

            train_count = (train_values == level).sum()
            test_count = (test_values == level).sum()

            train_percentage = (
                train_count / len(train_df) * 100
            )

            test_percentage = (
                test_count / len(test_df) * 100
            )

            results.append({
                "Variables": variable,
                "Level": str(level),
                "Training set": (
                    f"{train_count} ({train_percentage:.1f}%)"
                ),
                "Testing set": (
                    f"{test_count} ({test_percentage:.1f}%)"
                ),
                "p": p_text if i == 0 else ""
            })

    table_2 = pd.DataFrame(results)

    display(table_2)

    table_2.to_csv(
        "Table_2_Training_Testing_Distribution.csv",
        index=False
    )

    print("Table 2 saved successfully.")

    return table_2


table_2 = generate_table_2(train_df, test_df)


# ============================================================
# 5. DOWNLOADABLE OUTPUT FILES
# ============================================================

print("\nBoth tables have been generated.")
print("1. Table_1_Baseline_Characteristics.csv")
print("2. Table_2_Training_Testing_Distribution.csv")


Dataset loaded successfully!
Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



TABLE 1: BASELINE CHARACTERISTICS
Non-diabetes patients: 500
Diabetes patients: 268


,Variables,Non-diabetes,Diabetes,p
0,Pregnancy,"2.00 (1.00, 5.00)","4.00 (1.75, 8.00)",<0.001
1,Glucose,"107.00 (93.00, 125.00)","140.00 (119.00, 167.00)",<0.001
2,BP,"70.00 (62.00, 78.00)","74.00 (66.00, 82.00)",<0.001
3,Skin,"21.00 (0.00, 31.00)","27.00 (0.00, 36.00)",0.013
4,Insulin,"39.00 (0.00, 105.00)","0.00 (0.00, 167.25)",0.066
5,BMI,"30.05 (25.40, 35.30)","34.25 (30.80, 38.78)",<0.001
6,Pedigree,"0.34 (0.23, 0.56)","0.45 (0.26, 0.73)",<0.001
7,Age,"27.00 (23.00, 37.00)","36.00 (28.00, 44.00)",<0.001


Table 1 saved successfully.

Training set size: 537
Testing set size: 231

TABLE 2: TRAINING VS TESTING DISTRIBUTION


,Variables,Level,Training set,Testing set,p
0,Pregnancies,High,120 (22.3%),49 (21.2%),0.563
1,Pregnancies,Medium,127 (23.6%),48 (20.8%),
2,Pregnancies,Low,290 (54.0%),134 (58.0%),
3,Glucose,High,136 (25.3%),56 (24.2%),0.935
4,Glucose,Low,148 (27.6%),66 (28.6%),
5,Glucose,Medium,253 (47.1%),109 (47.2%),
6,Blood Pressure,Normal,400 (74.5%),172 (74.5%),0.035
7,Blood Pressure,Low,117 (21.8%),41 (17.7%),
8,Blood Pressure,High,20 (3.7%),18 (7.8%),
9,Skin Thickness,High,166 (30.9%),72 (31.2%),0.862


Table 2 saved successfully.

Both tables have been generated.
1. Table_1_Baseline_Characteristics.csv
2. Table_2_Training_Testing_Distribution.csv
